# Superstore Sales - Exploratory Data Analysis

先看看数据长什么样，了解一下基本的分布和趋势。

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from src.data_loader import load_raw_data, get_monthly_sales, get_category_monthly

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

%matplotlib inline

## 1. 加载数据

In [ ]:
df = load_raw_data('../data/raw/train.csv')
print(f'数据量: {len(df)} rows')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# 看看日期范围
print(f"Order Date range: {df['Order Date'].min()} ~ {df['Order Date'].max()}")
print(f"\nCategories: {df['Category'].unique()}")
print(f"Regions: {df['Region'].unique()}")

## 2. 月度销售趋势

先聚合到月度看整体走势

In [ ]:
monthly = get_monthly_sales(df)
monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly.values, marker='o', markersize=3, color='steelblue')
ax.set_title('Monthly Sales Trend')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

能看到一个明显的上升趋势，而且每年年底都有一个peak，应该是holiday season的效应。

## 3. 按品类拆分看看

In [ ]:
cat_monthly = get_category_monthly(df)

fig, ax = plt.subplots(figsize=(14, 5))
for cat in cat_monthly['category'].unique():
    subset = cat_monthly[cat_monthly['category'] == cat]
    ax.plot(subset['date'], subset['sales'], label=cat, marker='o', markersize=2)
ax.legend()
ax.set_title('Monthly Sales by Category')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/category_trend.png', dpi=150, bbox_inches='tight')
plt.show()

Technology品类波动最大，Furniture居中，Office Supplies相对稳定。

## 4. 区域分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# sales by region
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=True)
region_sales.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Total Sales by Region')
axes[0].set_xlabel('Sales ($)')

# sales by category
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=True)
cat_sales.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Total Sales by Category')
axes[1].set_xlabel('Sales ($)')

plt.tight_layout()
plt.savefig('../results/figures/region_category_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 季节性分解 (STL)

用STL把trend、seasonal、residual拆开看看。

In [ ]:
stl = STL(monthly, period=12)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.savefig('../results/figures/stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

趋势很明显在上升，季节性也很规律，residual里没有太明显的pattern，说明分解效果还行。

## 6. 自相关分析 (ACF / PACF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(monthly, lags=24, ax=axes[0])
axes[0].set_title('ACF')

plot_pacf(monthly, lags=24, ax=axes[1], method='ywm')
axes[1].set_title('PACF')

plt.tight_layout()
plt.savefig('../results/figures/acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

ACF缓慢衰减说明有趋势，lag 12处有spike说明有年度季节性。这跟前面STL的结论一致。

## 7. 平稳性检验 (ADF Test)

In [ ]:
def adf_test(series, name=''):
    result = adfuller(series, autolag='AIC')
    print(f'ADF Test: {name}')
    print(f'  ADF Statistic: {result[0]:.4f}')
    print(f'  p-value: {result[1]:.4f}')
    print(f'  Stationary: {"Yes" if result[1] < 0.05 else "No"}')
    print()

adf_test(monthly, 'Original Series')
adf_test(monthly.diff().dropna(), 'First Difference')
adf_test(monthly.diff(12).dropna(), 'Seasonal Difference (lag=12)')

原始序列不平稳（意料之中），一阶差分后就平稳了。ARIMA的d=1应该够了。

## 8. 月度销售箱线图

再看看每个月的分布，确认一下季节性

In [ ]:
monthly_df = pd.DataFrame({'sales': monthly})
monthly_df['month'] = monthly_df.index.month

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=monthly_df, x='month', y='sales', ax=ax, color='lightblue')
ax.set_title('Sales Distribution by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('../results/figures/monthly_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

9月和11-12月是销售高峰，1-2月是低谷。跟零售行业的直觉吻合。

---

**小结**: 数据有明显的上升趋势和年度季节性，一阶差分可以平稳化。接下来可以上SARIMA和Prophet了。